In [ ]:
# dependencies: a recent pytorch and %pip install transformers==4.46.3 datasets==2.20.0 accelerate==0.32.1
# the assignment may still work on other recent versions (transformers>4.25 and datasets>2.9), but no promises


### Part 1: Memory-efficient training and inference

__Your quest__ is to fine-tune a large language with restricted GPU memory. You can choose one of these two models:

- colab, kaggle or datasphere: choose either [facebook/opt-6.7b](https://huggingface.co/facebook/opt-6.7b), [Qwen/Qwen2.5-7B](https://huggingface.co/Qwen/Qwen2.5-7B) or Llama-3-8B ([official](https://huggingface.co/meta-llama/Llama-3.1-8B), [unsloth](https://huggingface.co/unsloth/Llama-3.1-8B))
- if you have >64GB disk space: [facebook/opt-iml-30b](https://huggingface.co/facebook/opt-iml-max-30b) or [Qwen/Qwen-32B](https://huggingface.co/Qwen/Qwen2.5-32B)

Both are powerful language models: opt-6.7b is a relatively old open-access GPT3 equivalent and Llama-3 / Qwen 2.5 are state-of-the-art LMs

You can use __up to 10GiB GPU memory__ (as in 3080 or 2080Ti) for 6.7B model and up to 48GB for the 30B one. We deliberately limit GPU memory below and recommend you to check the peak memory usage via: [`torch.cuda.max_memory_allocated()`](https://pytorch.org/docs/stable/generated/torch.cuda.max_memory_allocated.html). We shall also assume that you don't have enough RAM to load the full model on CPU. If your your machine has enough, you may take advantage of it.


Your code should be able to do 3 things:
* run forward pass on a sequence of 2048 tokens
* compute gradients w.r.t. a small subset of parameters: only one layer or similar
* generate an answer to a question using `model.generate` (see below)


Model compression alone will not count for full grade! Please either use a 16-bit model (6-8B) or, if you feel like you want to prune/quantize the model, use the 30B+ version.

In [2]:
import torch
# if your GPU has less than 10GB memory, please remove the code below
# if your GPU has less than 4GB memory, use colab or kaggle instead
max_memory_gib = torch.cuda.get_device_properties('cuda').total_memory / 2 ** 30
torch.cuda.set_per_process_memory_fraction(min(1.0, 10 / max_memory_gib))
print(f"Setting memory limit to {min(1.0, 11 / max_memory_gib) * 100:.2f}%")

Setting memory limit to 100.00%


For now, we're gonna load a smaller version of the model to show you around.

The large models use the same code, but with more layers & hidden units - so you can debug your code on the smaller model, then switch to the real deal.

In [30]:
import transformers
model_name = "facebook/opt-iml-1.3b"  # full model: 'facebook/opt-6.7b' or see above
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
model = transformers.AutoModelForCausalLM.from_pretrained(
    model_name, low_cpu_mem_usage=True, torch_dtype=torch.float16).cuda()

model.enable_input_require_grads()  # for gradient checkpointing compatibility, see FAQ

OutOfMemoryError: CUDA out of memory. Tried to allocate 32.00 MiB. GPU 0 has a total capacity of 5.64 GiB of which 46.94 MiB is free. Process 12051 has 70.34 MiB memory in use. Including non-PyTorch memory, this process has 5.18 GiB memory in use. Of the allocated memory 4.67 GiB is allocated by PyTorch, and 380.53 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

### Inference baseline

Here's a simple code that generates some tokens without offloading. You can use this as a reference, to check that your offloading algorithm is correct. Naturally, it will not work on the full 6.7B (or 30B) model.

In [10]:
# here's how the model works: tokenizer converts raw data to pytorch tensors
batch = tokenizer(["A cat sat", "import numpy"], return_tensors='pt')
batch = {name: tensor.cuda() for name, tensor in batch.items()}
print("Batch:", repr(batch)[:70].replace('\n', ' '), ' ...')


Batch: {'input_ids': tensor([[    2,   250,  4758,  4005],         [    2, 41  ...


In [4]:
# fun fact: you can use the model to generate text given prefix
generated_ids = model.generate(**batch, max_length=32)
print("Sample A:", tokenizer.decode(generated_ids[0]))
print("Sample B:", tokenizer.decode(generated_ids[1]))

Sample A: </s>A cat sat on my lap and I was watching a movie. I was about to fall asleep and the cat jumped up and started licking my face. I
Sample B: </s>import numpy.array

import numpy as np

import matplotlib.pyplot as plt

import numpy.array


### Training baseline

Here's some sample data you can use for prototyping -- and demonstrating that your algorithm works.
Then again, you are free to use any dataset you like.

We also provide a very simple fine-tuning example that mimics [BitFit](https://arxiv.org/abs/2106.10199).

In [5]:
from datasets import load_dataset

data = load_dataset("wikitext", "wikitext-2-v1")['train']
tokenizer.pad_token = tokenizer.eos_token

sample_batch = tokenizer(data['text'][:1], max_length=5, padding=True, pad_to_multiple_of=5, return_tensors='pt')

# note: sample_batch has a size of 1x5, you will need a larger batch in the next assignment
# note(2) if you want something more peculiar, https://huggingface.co/datasets/transformersbook/codeparrot

/home/dali/prsnl/efficient-dl-systems/.venv/lib/python3.13/site-packages/transformers/tokenization_utils_base.py:2852: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


In [9]:
# example: only train bias parameters, as in the BitFitPaper
for name, param in model.named_parameters():
    param.requires_grad = name.endswith("bias")
    if param.requires_grad:
        param.data = param.data.to(torch.float32)
print(f"Total parameters: {sum(p.numel() for p in model.parameters())/1e6:0.2f} million")
print(f"Trained parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:0.2f} million")


opt = torch.optim.Adam(model.parameters(), lr=1e-4)
# model turns those tensors into logits (pre-softmax activations) and loss
# in the example below, logits are available as pred.logits, and loss is pred.loss

for i in range(10):
    sample_batch = {name: tensor.cuda() for name, tensor in sample_batch.items()}
    with torch.cuda.amp.autocast():
        loss = model(**sample_batch, labels=sample_batch['input_ids']).loss / 1000
    loss.backward()
    opt.step()
    print(f"Loss[{i}] = {loss.item():.3f}")

# if all went well, you'll see the loss go down

Total parameters: 1315.76 million
Trained parameters: 0.54 million


/tmp/ipykernel_16350/3588210229.py:16: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Loss[0] = 0.011
Loss[1] = 0.010
Loss[2] = 0.009
Loss[3] = 0.008
Loss[4] = 0.007
Loss[5] = 0.007
Loss[6] = 0.006
Loss[7] = 0.005
Loss[8] = 0.005
Loss[9] = 0.004


If it looked a bit too easy - that was because you are dealing with a small model that fits into RAM. Once you have something larger, you can no longer simply `.from_pretrained` your model. Instead, you will need to process weights in small groups - the way they are stored in Hugging Face hub.

## Assignment details


Your main objective is to implement parameter offloading and solve two problems: fine-tuning and inference.

__Task 1.1:__ run forward and backward pass, accumulate gradients w.r.t. a subset of model parameters. Use a training batch size of 128 sequences, and sequence length of 1024 tokens. In other words, `input_ids.shape == (128, 1024)`.

You may choose one of these options:
- train only the embedding layer, [similar to this paper](https://arxiv.org/abs/2104.08691)
- train low-rank adapters (LoRA), [like in this paper](https://arxiv.org/abs/2106.09685)
- use [Hugging Face PEFT](https://github.com/huggingface/peft/)

You don't have to train the model to convergence, just show that it can run 10 consecutive forward-backward-step passes and **the loss goes down**. You can even run those forward/backward passes on the same batch!


Please do not use native offloading / quantization libraries for this assignment - you are to implement your own.


__Task 1.2:__ generate a short sequence given a prefix. You may choose any generation task that requires generating at least 25 consecutive tokens. Here's one example from the NLP course (the generated code is in blue)

![img](https://i.imgur.com/a1QhKF7.png)

You may use model.generate (if your code is compatible with that) or write your own inference loop. If you choose to write your own loop, you are free to use sampling, greedy, top-p, top-k or any other [inference mode supported by HF transformers](https://huggingface.co/docs/transformers/main_classes/text_generation).


__Grading (5 points):__

- __+1 point__ you can perform forward pass with offloading on *some* input sequence (any batch size / length)
- __+1 point__ check that forward pass with offloading is `torch.allclose` to forward pass without offloading
    - since you (likely) can't run the full model w/o offloading, test it the 1.3B model from earlier
- __+1 point__ you can perform forward pass on 128x1024 tokens of actual text data (e.g. the sample data above)
- __+1 point__ you can compute gradients with offloading on the same 128x1024 tokens from the real text data
- __+1 point__ you can inference the model - and it generates some human-readable text
- __bonus points:__ we offer two optional assignments:
   - **Selective activation checkpointing (2pt):** there is a gentler version of gradient checkpointing where you don't just remember the layer inputs, but also some activations that are easier to compute - compared to their size. For instance, MLP linear layers are compute-heavy, but the nonlinearity is relatively compute-light for the same amount of memory. You can re-compute only the compute-light operations and keep the compute-heavy ones in memory. There's [a paper](https://arxiv.org/pdf/2205.05198) that describes such an approach in detail (see 'Selective activation checkpointing').
   - **Prefetch offloaded layers (2pt):** optimize your code so that it begins pre-loading the next offloaded layer in the background, while computing the current layer. It can be done with a copy with non_blocking=True, or, for fine-grained control, CUDA streams. To get the full grade for this assignment, please demonstrate that your approach is faster than naive offloading, at least during large batch forward/backward pass. This can be done using a profiler.
   - Please note that the maximum points for this week are **capped at 14**.

__Conditions:__
- using more than 10GiB of GPU memory at any point is forbidden (check with [`torch.cuda.max_memory_allocated()`](https://pytorch.org/docs/stable/generated/torch.cuda.max_memory_allocated.html))
- please keep all model parameters in either float16, bfloat16, or float32 - no quantization for now
   - if you *really* want to show off quantization, evaluate your code with both original and quantized weights
- at least 99% of model's floating point computations should be done on GPU. If you find a server with a ton of RAM and run the model on cpu, it will not count as a solution
- please do **not** use any thrid-party offloading implementations (e.g. from deepspeed or accelerate)
- your solution may be slow - especially when loading from colab disks. This is not your fault :)
   - if you found a way to speed up the code in a non-trivial way (e.g. load i+1st layer in parallel when computing i-th), please attach a short summary of what when submitting the notebook (e.g. anytask/lms) to get bonus points



__FAQ:__

- __My training outputs have .requires_grad == False!__ This may be a side-effect of using gradient checkpointing if all your trainable parameters are inside the checkpoints (e.g. with LoRA). To circumvent this, either set `model.enable_input_require_grads()` or manually ensure that input tensors to each checkpoint have requires_grad=True.

- __I am getting out-of-memory errors for no reason!__
  - it could be because of some leftover tensors from previous cells. To get rid of them, please restart the notebook and only run the code that is relevant to your current task.

- __The forward pass activations are too large, it does not fit!__
   - __Gradient accumulation:__ you probably can't process 128 sequences at once -- but what if you accumulate them over several forward/batckward passes with a smaller batch size.
   - __Gradient checkpointing:__ you can further reduce activation memory by not storing intermediate activations. You can learn how to usa built-in checkpoints [from their docs](https://huggingface.co/docs/transformers/main_classes/model) or build your own using [PyTorch default checkpointing](https://pytorch.org/docs/stable/checkpoint.html).
  
- __My float16 gradients are NaN!__
   - There should be a way to scale your loss function by a constant -- only to un-scale it later. You can use GradScaler from [PyTorch AMP](https://pytorch.org/docs/stable/amp.html) or write your own monstrosity.
   - You can also cast weights to bfloat16 _but you have to demonstrate that bfloat16 model generates the same (or close) output as float16 one!_ As in "you have to write a short report with code and samples."
     
- __I can run forward with no_grad, but running with grad goes out of memory!__
   - If the problem only occurs with large batches, please see "activations are too large" above.
   - If you get OOM errors even with a single training token (a 1x1 batch), but only in training mode,
     maybe you forgot to mark most parameters as `requires_grad=False`? The .grad buffers can be quite large.
     
   - If not, OOM  be because PyTorch autograd remembers the intermediate weight tensors for backprop.
     For example, consider this code:
     
```python
    x = embeddings_and_input_layernorm(input_ids)
    for layer_index in range(num_layers):
        layer = load_from_disk(layer_index)
        x = layer(x)
        del layer  # we no longer need this layer's weights, but PyTorch will keep it in memory for autograd!
```

    If this is your case, you can write an autograd function that loads the necessary weight.
    a look at "[Optional] Suggested Interface" section below.
     
- __I cannot load the full model even in CPU RAM!__
   - This is intended - and a real problem that you often face in production.
     You gotta find a way to prepare your model for offloading without loading the full thing into RAM.
     In the next section, we explain how you can handle checkpoints and initialize the model in google colab.
     Please see the [Optional] sections that mention low RAM.


<details>
    <summary> <h3> <u> [Optional] Suggested Interface with torch.autograd.Function (click to expand) </u> </h3> </summary>

You can assume that offloaded weights do not require grad themselves - but they take part in intermediate computations that *do* require grad.
The problem is, if you load weights naively without `torch.no_grad`, PyTorch will remember them until the end of backward pass. If not addressed, this will keep all model weights in memory and mess up your offloading.


To avoid this, you can implement a custom autograd function that loads weights from ram / disk internally. That way, PyTorch will not keep any gpu tensors except unless you explicitly tell it to. Crucially, __we only need this function for linear layers__ since all other layers can fit on GPU. Though, you may *optionally* offload embedding layers as well.


Here's [some documentation](https://pytorch.org/docs/stable/notes/extending.html#extending-torch-autograd) on writing your own autograd functions. Your solution could look something like this:


```python
class _OffloadedLinearOp(torch.autograd.Function):
    @staticmethod
    def forward(ctx, input, saved_weight_path, bias_or_none):
        weight = you.load_by_name(saved_weight_path)
        ctx._saved_weight_path = saved_weight_path
        ctx._has_bias = bias_or_none is not None
        return torch.nn.functional.linear(input, weight, bias=bias_or_none)

    @staticmethod
    def backward(ctx, grad_output):
        weight = you.load_by_name(ctx._saved_weight_path)
        grad_input = torch.nn.functional.linear(grad_output, weight.t())
        grad_bias = grad_output.flatten(0, -2).sum(0) if ctx._has_bias else None
        return grad_input, None, grad_bias

    
# to use:
# output = _OffloadedLinearOp.apply(input, "my_weight.pth", bias)
# loss(output).backward()  # uses custom backward
```

You can implement this function separately and test it on a single layer to make sure forward and backward passes match. Once you are confident in your code, it's time to apply it to your model. One way to do this is:


```python
class MyOffloadedLinear(torch.nn.Module):
    def __init__(self, saved_weight_path, bias_or_none):
        super().__init__()
        self.saved_weight_path, self.bias_or_none = saved_weight_path, bias_or_none
    def forward(self, input):
        return _OffloadedLinearOp.apply(input, self.saved_weight_path, self.bias_or_none)

for module_that_contains_linear in you.find_these_modules(model):
    linear = you.take_linear_layer_from(module_that_contains_linear)
    saved_weight_path = save_weight_somewhere(linear.weight)
    offloaded_linear = MyOffloadedLinear(saved_weight_path, linear.bias)
    you.replace_that_linear_with(offloaded_linear)
```
    
Please note that this algorithm is "lazy" in the sense that it loads weights just in time. A smarter (and faster!) way to offload the data is to do it in parallel: once you load the first weight, you immediately start loading the second weight from disk in a background thread. You can do this by recording the order in which your model uses the offloaded weights and keeping track of which weight you should load next.

</details>

<details>
    <summary><h3><u>[Optional] How to initialize the model with low RAM (click to expand)</u></h3></summary>
    
    The trick is that you don't initialize all modules at once.
    Instead, you can load *some* modules, prepare them for offloading (e.g. remove some params), then load the next bunch of modules.
    
    Here's one way you can do this:

    ```python
    config = transformers.AutoConfig.from_pretrained("facebook/opt-6.7b")
    actual_hidden_layers = config.num_hidden_layers
    config.num_hidden_layers = 0  # create a model with no hidden layers
    model = transformers.AutoModelForCausalLM.from_config(config, torch_dtype=torch.float16)
    print(f"Total parameters (embeddings only): {sum(p.numel() for p in model.parameters())/1e6:0.2f} million")
    # only 0.21 billion instead of 6.7

    for _ in range(actual_hidden_layers):
        new_layer = transformers.models.opt.modeling_opt.OPTDecoderLayer(config)
        new_layer = you.prepare_for_offloading(new_layer)
        model.model.decoder.layers.append(new_layer)
    config.num_hidden_layers = actual_hidden_layers

    you.load_parameters_that_werent_offloaded(model, preprocessed_checkpoint_chunks)
    ```
    
    If `you.prepare_for_offloading` properly offloads all heavy parameters to the disk, this code will build the full offloaded model without going over 10GB CPU RAM.
    We also recommend that you check that the resulting code works correctly by test-running it on the 1.3B model.

</details>


<details>
    <summary><h3><u>[Optional] Dealing with HuggingFace weights with low RAM (click to expand)</u></h3></summary>


When you download a Hugging Face model, there will be one or more "chunks", holding the data parameters. These chunks can be seen in the model repository, under "Files and versions" tab:
![image.png](https://i.imgur.com/3gZ2KPB.png)

Reference links to "files and versions": [opt-6.7b](https://huggingface.co/facebook/opt-6.7b/tree/main), [opt-iml-30b](https://huggingface.co/facebook/opt-iml-30b/tree/main)

You can download individual chunks of parameters by going clicking on a chunk and copying the "download" url, like this:

![img](https://i.imgur.com/cv9WvYw.png)

Any chunks downloaded this way will contain a `torch.load`-able state dict. Here's how it works:

```python
# example: download one (small) chunk out of OPT-IML-30B
chunk7_download_url = "https://huggingface.co/facebook/opt-iml-30b/resolve/828fabfb08d5d3f81b4d33cd27a64e3a360a5770/pytorch_model-00007-of-00007.bin"
!wget {chunk7_download_url} -O "chunk7.pth"

partial_state_dict = torch.load("chunk7.pth")
print(f"Keys:", partial_state_dict.keys(), '\n')
print(f"Shape of decoder.layers.47.fc1.weight: {partial_state_dict['decoder.layers.47.fc1.weight'].shape}")
# Keys: dict_keys(['decoder.layers.47.fc1.weight', 'decoder.layers.47.fc1.bias', 'decoder.layers.47.fc2.weight', 'decoder.layers.47.fc2.bias', 'decoder.layers.47.final_layer_norm.weight', 'decoder.layers.47.final_layer_norm.bias'])
# Shape of decoder.layers.47.fc1.weight: torch.Size([28672, 7168])
```

</details>


In [6]:
import gc
import torch
import torch.nn as nn
import torch.nn.functional as F

WEIGHT_STORE = {}   # {weight_key: cpu_tensor}


class OffloadedLinearFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, input, weight_key, bias):
        weight = WEIGHT_STORE[weight_key].to(input.device) # CPU → GPU
        ctx._weight_key = weight_key
        ctx._has_bias = bias is not None
        output = F.linear(input, weight, bias)
        del weight # free GPU copy
        return output

    @staticmethod
    def backward(ctx, grad_output):
        weight = WEIGHT_STORE[ctx._weight_key].to(grad_output.device)
        grad_input = grad_output.matmul(weight)
        grad_bias = (
            grad_output.flatten(0, -2).sum(0) if ctx._has_bias else None
        )
        del weight
        return grad_input, None, grad_bias


class OffloadedLinear(nn.Module):
    """Drop-in nn.Linear replacement — weight on CPU, bias on GPU."""

    def __init__(self, weight_key, bias=None):
        super().__init__()
        self.weight_key = weight_key
        self.bias = bias

    def forward(self, input):
        return OffloadedLinearFn.apply(input, self.weight_key, self.bias)


def offload_model_linear_layers(model):
    n = 0
    targets = {"q_proj", "k_proj", "v_proj", "out_proj", "fc1", "fc2"}

    for name, module in list(model.named_modules()):
        for attr in targets:
            child = getattr(module, attr, None)
            if isinstance(child, nn.Linear):
                key = f"{name}.{attr}.weight"
                WEIGHT_STORE[key] = child.weight.data.cpu() # store on CPU
                bias = (
                    nn.Parameter(child.bias.data.clone())
                    if child.bias is not None
                    else None
                )
                setattr(module, attr, OffloadedLinear(key, bias))
                n += 1

    gc.collect()
    torch.cuda.empty_cache()
    mb = sum(t.numel() * t.element_size() for t in WEIGHT_STORE.values()) / 1e6
    print(f"Offloaded {n} linear layers ({mb:.0f} MB) to CPU")
    return model


print("Offloading infrastructure ready")

Offloading infrastructure ready


In [11]:
torch.cuda.reset_peak_memory_stats()

test_input = tokenizer("The meaning of life is", return_tensors="pt")
test_input = {k: v.cuda() for k, v in test_input.items()}

# Baseline
with torch.no_grad():
    baseline_logits = model(**test_input).logits.clone()

# Apply offloading
offload_model_linear_layers(model)

# Forward again
with torch.no_grad():
    offloaded_logits = model(**test_input).logits

# Compare
max_diff = (baseline_logits - offloaded_logits).abs().max().item()
print(f"Max absolute difference: {max_diff}")
assert torch.allclose(baseline_logits, offloaded_logits, atol=1e-3), f"MISMATCH {max_diff}"

print(f"Peak GPU memory: {torch.cuda.max_memory_allocated() / 2**30:.2f} GiB")

Offloaded 144 linear layers (2416 MB) to CPU
Max absolute difference: 0.0
Peak GPU memory: 2.46 GiB


In [12]:
model

OPTForCausalLM(
  (model): OPTModel(
    (decoder): OPTDecoder(
      (embed_tokens): Embedding(50272, 2048, padding_idx=1)
      (embed_positions): OPTLearnedPositionalEmbedding(2050, 2048)
      (final_layer_norm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
      (layers): ModuleList(
        (0-23): 24 x OPTDecoderLayer(
          (self_attn): OPTSdpaAttention(
            (k_proj): OffloadedLinear()
            (v_proj): OffloadedLinear()
            (q_proj): OffloadedLinear()
            (out_proj): OffloadedLinear()
          )
          (activation_fn): ReLU()
          (self_attn_layer_norm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
          (fc1): OffloadedLinear()
          (fc2): OffloadedLinear()
          (final_layer_norm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
        )
      )
    )
  )
  (lm_head): Linear(in_features=2048, out_features=50272, bias=False)
)

In [13]:
from datasets import load_dataset

data = load_dataset("wikitext", "wikitext-2-v1")["train"]
tokenizer.pad_token = tokenizer.eos_token

all_text = "\n".join(t for t in data["text"] if t.strip())
all_tokens = tokenizer(all_text, return_tensors="pt", truncation=False)["input_ids"].squeeze(0)

seq_len = 1024
n_seq = len(all_tokens) // seq_len
train_ids = all_tokens[: n_seq * seq_len].reshape(n_seq, seq_len)[:128]
print(f"Training data: {train_ids.shape}")            

Training data: torch.Size([128, 1024])


In [14]:
# BitFit: freeze everything, unfreeze biases only
for name, p in model.named_parameters():
    p.requires_grad = name.endswith("bias")
    if p.requires_grad:
        p.data = p.data.to(torch.float32)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable/1e6:.2f}M / {total_params/1e6:.2f}M total")

Trainable: 0.54M / 107.80M total


In [15]:
# Gradient checkpointing
model.gradient_checkpointing_enable()
model.enable_input_require_grads()
model.train()

OPTForCausalLM(
  (model): OPTModel(
    (decoder): OPTDecoder(
      (embed_tokens): Embedding(50272, 2048, padding_idx=1)
      (embed_positions): OPTLearnedPositionalEmbedding(2050, 2048)
      (final_layer_norm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
      (layers): ModuleList(
        (0-23): 24 x OPTDecoderLayer(
          (self_attn): OPTSdpaAttention(
            (k_proj): OffloadedLinear()
            (v_proj): OffloadedLinear()
            (q_proj): OffloadedLinear()
            (out_proj): OffloadedLinear()
          )
          (activation_fn): ReLU()
          (self_attn_layer_norm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
          (fc1): OffloadedLinear()
          (fc2): OffloadedLinear()
          (final_layer_norm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
        )
      )
    )
  )
  (lm_head): Linear(in_features=2048, out_features=50272, bias=False)
)

In [16]:
# Training loop with gradient accumulation
micro_batch = 4                          
n_accum = train_ids.shape[0] // micro_batch
opt = torch.optim.Adam(
    [p for p in model.parameters() if p.requires_grad], lr=1e-4
)

In [18]:
torch.cuda.reset_peak_memory_stats()

for step in range(10):
    opt.zero_grad()
    total_loss = 0.0

    for i in range(n_accum):
        mb_ids = train_ids[i * micro_batch : (i + 1) * micro_batch].cuda()
        with torch.amp.autocast("cuda"):
            loss = model(input_ids=mb_ids, labels=mb_ids).loss / n_accum
        loss.backward()
        total_loss += loss.item()
        del mb_ids, loss

    opt.step()
    torch.cuda.empty_cache()
    print(f"Step {step:2d} | loss = {total_loss:.4f}")

print(f"Peak GPU memory: {torch.cuda.max_memory_allocated() / 2**30:.2f} GiB")

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


Step  0 | loss = 2.8693
Step  1 | loss = 2.8580
Step  2 | loss = 2.8497
Step  3 | loss = 2.8414
Step  4 | loss = 2.8369
Step  5 | loss = 2.8316
Step  6 | loss = 2.8282
Step  7 | loss = 2.8240


KeyboardInterrupt: 

In [ ]:
model = model.half()
model.gradient_checkpointing_disable()
model.eval()
torch.cuda.reset_peak_memory_stats()

prompts = [
    "The theory of relativity states that",
    "Once upon a time in a land far away,",
    "def quicksort(arr):\n",
]

for prompt in prompts:
    input_ids = tokenizer(prompt, return_tensors="pt")["input_ids"].cuda()
    with torch.no_grad():
        generated = model.generate(
            input_ids,
            max_new_tokens=50,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
        )
    text = tokenizer.decode(generated[0], skip_special_tokens=True)
    print(f"Prompt : {prompt!r}")
    print(f"Output : {text}")
    print("—" * 60)

print(f"Peak GPU memory: {torch.cuda.max_memory_allocated() / 2**30:.2f} GiB")

Prompt : 'The theory of relativity states that'
Output : The theory of relativity states that time and space are relative.  The Earth is in the center of the universe and the sun is the center of the solar system.  The Sun is in the center of the galaxy and the galaxy is in the center of the universe.  The
————————————————————————————————————————————————————————————
Prompt : 'Once upon a time in a land far away,'
Output : Once upon a time in a land far away, the hero of the story was a prince who grew up in a castle. One day the prince came home and told his parents that he wanted to go to the kingdom to see the queen. He was so excited that he told his father that he was
————————————————————————————————————————————————————————————
Prompt : 'def quicksort(arr):\n'
Output : def quicksort(arr):

arr[0] = sorted(all)

def quicksort(arr):

arr[0] = sorted(all)

def quicksort(arr):

arr[0] = sorted(all)
————————————————————————————————————————————————————————————

Generation complete!
Peak 

In [21]:
print(f"Peak GPU memory:     {torch.cuda.max_memory_allocated() / 2**30:.2f} GiB")
print(f"Current GPU memory:  {torch.cuda.memory_allocated() / 2**30:.2f} GiB")
print(f"CPU weight store:    {sum(t.numel()*t.element_size() for t in WEIGHT_STORE.values()) / 2**30:.2f} GiB")
print(f"Offloaded tensors:   {len(WEIGHT_STORE)}")

Peak GPU memory:     0.28 GiB
Current GPU memory:  0.22 GiB
CPU weight store:    2.25 GiB
Offloaded tensors:   144


## Gradient Checkpointing from Scratch

In [7]:
class CheckpointFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, input, layer):
        ctx.save_for_backward(input.detach())
        ctx.layer = layer

        with torch.no_grad():
            output = layer(input)
        return output.detach().requires_grad_(input.requires_grad)

    @staticmethod
    def backward(ctx, grad_output):
        (input,) = ctx.saved_tensors
        input = input.detach().requires_grad_(True)

        with torch.enable_grad():
            output = ctx.layer(input)

        # Backprop through local graph
        torch.autograd.backward(output, grad_output)

        return input.grad, None 


def checkpoint(layer, input):
    return CheckpointFunction.apply(input, layer)


print("Testing custom grad checkpointing...")
test_linear = torch.nn.Linear(64, 64, device="cuda", dtype=torch.float32)
x = torch.randn(2, 64, device="cuda", requires_grad=True)

# Normal forward-backward
y_normal = test_linear(x)
loss_normal = y_normal.sum()
loss_normal.backward()
grad_normal = x.grad.clone()

x.grad = None

# Checkpointed forward-backward
y_ckpt = checkpoint(test_linear, x)
loss_ckpt = y_ckpt.sum()
loss_ckpt.backward()
grad_ckpt = x.grad.clone()

assert torch.allclose(y_normal, y_ckpt, atol=1e-6), "Forward mismatch!"
assert torch.allclose(grad_normal, grad_ckpt, atol=1e-5), "Backward mismatch!"
print("Custom checkpoint forward & backward match naive implementation!")

Testing custom grad checkpointing...
Custom checkpoint forward & backward match naive implementation!


## Attention

In [2]:
import transformers
import torch

model_name = "facebook/opt-iml-1.3b"   # full model: 'facebook/opt-6.7b' or see above
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
model = transformers.AutoModelForCausalLM.from_pretrained(
    model_name, low_cpu_mem_usage=True, torch_dtype=torch.float16, attn_implementation="flash_attention_2")

model.enable_input_require_grads()

ImportError: FlashAttention2 has been toggled on, but it cannot be used due to the following error: the package flash_attn seems to be not installed. Please refer to the documentation of https://huggingface.co/docs/transformers/perf_infer_gpu_one#flashattention-2 to install Flash Attention 2.

In [8]:
from datasets import load_dataset

data = load_dataset("wikitext", "wikitext-2-v1")["train"]
seqs = [d['text'] for d in data]

/home/dali/prsnl/efficient-dl-systems/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [23]:
import random

def get_batch_packed(batch_size, seq_len):
    tokenizer.pad_token = tokenizer.eos_token

    batch_input_ids = []
    batch_attention_mask = []
    batch_position_ids = []

    for _ in range(batch_size):
        packed_ids = []
        position_ids = []
        segment_mask = []

        while True:
            text = random.choice(seqs)

            tokens = tokenizer(
                text,
                add_special_tokens=False,
                return_tensors="pt"
            )["input_ids"][0]

            if len(tokens) == 0:
                continue

            seg_len = len(tokens) + 1

            if len(packed_ids) + seg_len > seq_len:
                break
            packed_ids.extend(tokens.tolist())
            packed_ids.append(tokenizer.eos_token_id)

            position_ids.extend(range(len(tokens)))
            position_ids.append(len(tokens))

            segment_mask.extend([1] * seg_len)

        pad_len = seq_len - len(packed_ids)

        packed_ids += [tokenizer.pad_token_id] * pad_len
        position_ids += [0] * pad_len
        segment_mask += [0] * pad_len

        batch_input_ids.append(torch.tensor(packed_ids))
        batch_position_ids.append(torch.tensor(position_ids))
        batch_attention_mask.append(torch.tensor(segment_mask))

    return {
        "input_ids": torch.stack(batch_input_ids),
        "attention_mask": torch.stack(batch_attention_mask),
        "position_ids": torch.stack(batch_position_ids),
    }

In [24]:
get_batch_packed(4, 512)

{'input_ids': tensor([[  497,   103,   477,  ...,     2,     2,     2],
         [  374,   211,   787,  ...,     2,     2,     2],
         [ 9732,   999, 16283,  ...,     2,     2,     2],
         [ 5300,  2843,  6938,  ...,     2,     2,     2]]),
 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0]]),
 'position_ids': tensor([[0, 1, 2,  ..., 0, 0, 0],
         [0, 1, 2,  ..., 0, 0, 0],
         [0, 1, 2,  ..., 0, 0, 0],
         [0, 1, 2,  ..., 0, 0, 0]])}

In [26]:
import torch.nn.functional as F
import transformers.models.opt.modeling_opt as opt_module
from transformers.modeling_flash_attention_utils import _flash_attention_forward

setattr(opt_module, "_flash_attention_forward", _flash_attention_forward)

decoder = model.model.decoder
layers = decoder.layers
embed_tokens = decoder.embed_tokens
embed_positions = decoder.embed_positions
final_layer_norm = decoder.final_layer_norm
lm_head = model.lm_head

In [27]:
class OffloadFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, layer, hidden_states, attention_mask, position_ids):
        ctx.layer = layer
        ctx.attention_mask = attention_mask
        ctx.position_ids = position_ids
        ctx.gpu_rng_state = torch.cuda.get_rng_state()
        ctx.save_for_backward(hidden_states)

        layer.to("cuda")
        with torch.no_grad():
            outputs = layer(hidden_states, attention_mask=attention_mask, position_ids=position_ids)
        layer.to("cpu", non_blocking=True)

        return outputs[0].detach().requires_grad_(hidden_states.requires_grad)

    @staticmethod
    def backward(ctx, grad_output):
        hidden_states, = ctx.saved_tensors
        hidden_states = hidden_states.detach().requires_grad_(True)
        layer = ctx.layer
        torch.cuda.set_rng_state(ctx.gpu_rng_state)

        layer.to("cuda")
        with torch.enable_grad():
            outputs = layer(hidden_states, attention_mask=ctx.attention_mask, position_ids=ctx.position_ids)[0]
            
        torch.autograd.backward(outputs, grad_output)
            
        layer.to("cpu", non_blocking=True)

        return None, hidden_states.grad, None, None

In [ ]:
from tqdm.auto import tqdm

def forward(batch):
    input_ids = batch["input_ids"].to("cuda")
    attention_mask = batch["attention_mask"].to("cuda")
    position_ids = batch["position_ids"].to("cuda")

    embed_tokens.to("cuda")
    embed_positions.to("cuda")
    
    inputs_embeds = embed_tokens(input_ids)
    pos_embeds = embed_positions(attention_mask=attention_mask, position_ids=position_ids)
    hidden_states = inputs_embeds + pos_embeds
    
    embed_tokens.to("cpu")
    embed_positions.to("cpu")

    for layer in tqdm(layers):
        hidden_states = OffloadFn.apply(
            layer, 
            hidden_states, 
            attention_mask, 
            position_ids
        )

    final_layer_norm.to("cuda")
    lm_head.to("cuda")
    
    hidden_states = final_layer_norm(hidden_states)
    logits = lm_head(hidden_states)
    return input_ids, logits

In [ ]:
@torch.no_grad()
def generate_from_text(text, max_new_tokens=20):
    inputs = tokenizer(text, return_tensors="pt", add_special_tokens=True)
    current_ids = inputs["input_ids"]

    for _ in range(max_new_tokens):
        seq_len = current_ids.size(1)

        current_batch = {
            "input_ids": current_ids,
            "attention_mask": torch.ones((1, seq_len), dtype=torch.long),
            "position_ids": torch.arange(seq_len).unsqueeze(0)
        }

        _, logits = forward(current_batch)

        next_token_logits = logits[:, -1, :]
        next_token = torch.argmax(next_token_logits, dim=-1).unsqueeze(-1)

        current_ids = torch.cat([current_ids, next_token.cpu()], dim=-1)

        if next_token.item() == tokenizer.eos_token_id:
            break

    return tokenizer.decode(current_ids[0], skip_special_tokens=True)

output_text = generate_from_text("The theory of relativity states that", max_new_tokens=30)
print(output_text)